# Appendix E: sensitivity to background size

Reproduce the saved B results for background size K, separately for MLP/FTT and Top-3 Jaccard, L2 and 1−RBO. Every point is **100 × count(distance > saved threshold) / (10N)** from all ten unordered pairs among five explanations. The saved threshold is fixed across K within each dataset/metric.

**Input:** `figures_b/background_sweep/pair_exceedance_counts.csv` (600 rows). K = 100 reuses the canonical main result, checked against `figures_b/background_table/pair_exceedance_counts.csv` (96 matching rows). No new experiments, baseline Monte Carlo, raw-distance calculations, or model/query-identity checks are performed here.

The x-axis uses actual K on a log2 scale. German Credit K = 790 and Diabetes K = 313 are full-training endpoints; ACS/GMSC stop at K = 1600. Segments connect observations without smoothing or confidence intervals. Coalition budgets remain as historically configured (German/GMSC: 1024; Diabetes/ACS: 256); they are not separately swept.

Run all cells with Python, NumPy, pandas, Matplotlib, IPython and Jupyter installed. The helper is `analysis/appendix_e/depth_sweep.py`; no machine-specific path is required.

**Current manuscript:** the two L2 plots are retained here as analysis records. The compact table in J18 replaces them in the paper; Jaccard and RBO plots remain included.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

# Locate this checkout when run from its root or the notebook directory.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / ".shap_multiplicity_root").is_file()
             and (p / "figures_b/background_table/pair_exceedance_counts.csv").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Run from this repository or one of its subdirectories.")
OUTPUT = REPO / "appendix_e"
HELPER_ROOT = REPO / "analysis"
sys.path.insert(0, str(HELPER_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "appendix-e-matplotlib"))

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from appendix_e.depth_sweep import (
    read_inputs, validate_topk, validate_sweep, export_analysis,
    plot_topk, plot_background_size, save_figure,
)
%matplotlib inline
print("Repository:", REPO)
print("Outputs:", OUTPUT)

## Validate the design, counts, thresholds and canonical K = 100

Check all 600 keys, finite values, integer counts, all ten pairs, the exact 10N denominator, and percentage reconstruction. Require one threshold per dataset/metric across all K, models and methods. The 96 K = 100 rows must exactly match main-table counts, rates and thresholds.

In [ ]:
sweep, main = read_inputs(REPO, "sweep")
sweep_report, sweep_equivalence = validate_sweep(sweep, main)
print(json.dumps(sweep_report, indent=2))
display(sweep.groupby(["dataset", "metric"], sort=False).agg(
    threshold=("threshold", "first"), threshold_count=("threshold", "nunique"),
    minimum_K=("bg_size", "min"), maximum_K=("bg_size", "max"),
    number_of_K=("bg_size", "nunique")))

## Export portable data and source hashes

Retain original counts, percentages and thresholds, and add method labels, actual-K/full-training flags, units, coalition-budget metadata and the K = 100 reuse policy. The validation sidecar records hashes of both source CSVs and the exported plotting CSV. These metadata describe the existing saved setting; no source experiment is rerun.

In [ ]:
sweep_provenance = export_analysis(REPO, OUTPUT, "sweep", sweep, sweep_report, sweep_equivalence)
sweep_plot = pd.read_csv(OUTPUT / "data/sweep_plot_data.csv", float_precision="round_trip")
print("Exported:", sweep_provenance["plot_data"]["path"])
display(sweep_plot[["dataset_label", "bg_size", "is_full_training", "requested_coalition_budget"]]
        .drop_duplicates().reset_index(drop=True))

## Primary metric: Top-3 Jaccard

The four dataset panels share a 0–100% y-axis. Method colors match the existing sampling figures, and distinct point markers show every observed K. Full-training endpoints are labeled at their true K positions. The distance threshold is not drawn on this percentage axis.

In [ ]:
fig = plot_background_size(sweep_plot, model="ftt", metric="jac")
assets = save_figure(fig, "appendix_background_ftt_jac")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

In [ ]:
fig = plot_background_size(sweep_plot, model="mlp", metric="jac")
assets = save_figure(fig, "appendix_background_mlp_jac")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

## Supporting metrics: retained L2 plots and manuscript RBO figures

The L2 plots below are retained for local inspection, but are no longer included in the manuscript. The manuscript uses the compact L2 table in J18. RBO is the existing full-depth extrapolated statistic with `p = 1 - 1/d`; this redraw neither changes its definition nor recomputes its baseline.


In [ ]:
fig = plot_background_size(sweep_plot, model="ftt", metric="l2")
assets = save_figure(fig, "appendix_background_ftt_l2")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

In [ ]:
fig = plot_background_size(sweep_plot, model="mlp", metric="l2")
assets = save_figure(fig, "appendix_background_mlp_l2")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

In [ ]:
fig = plot_background_size(sweep_plot, model="ftt", metric="rbo")
assets = save_figure(fig, "appendix_background_ftt_rbo")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

In [ ]:
fig = plot_background_size(sweep_plot, model="mlp", metric="rbo")
assets = save_figure(fig, "appendix_background_mlp_rbo")
plt.show()
plt.close(fig)
print(json.dumps(assets, indent=2))

## Read the observed pattern without imposing monotonicity

The table below retrieves the GMSC/FTT Random Top-3 observations at K = 100, 200 and 1600 directly from the plotting CSV. A larger endpoint K can reduce disagreement while intermediate K increases it. These observational results do not isolate a causal effect of background size or provide an independent estimate of coalition-sampling variability. Pairs from a common query/run are dependent; no unsupported standard-error bands are added.

In [ ]:
display(sweep_plot.loc[(sweep_plot["dataset"] == "gmsc")
                       & (sweep_plot["model"] == "ftt")
                       & (sweep_plot["method"] == "random")
                       & (sweep_plot["metric"] == "jac")
                       & sweep_plot["bg_size"].isin([100, 200, 1600]),
                       ["bg_size", "percent", "n_exceed", "n_observations", "threshold"]]
        .sort_values("bg_size").reset_index(drop=True))

## Compact L2 table for the manuscript

Each dataset/model row reports the minimum and maximum **B exceedance percentages across the four methods** at K=25 and K=50, followed by the **maximum across all four methods and all evaluated K>=100**. These are descriptive ranges and maxima of existing percentages; they are not confidence intervals, pooled percentages, or per-instance mean distances.

The final column includes the actual full-training endpoints for German/Diabetes and stops at K=1600 for ACS/GMSC. Zero exceedance means no observed pair distance exceeded the selected threshold; it does not mean zero distance or stable feature rankings. Export the exact eight-row CSV, LaTeX tabular and provenance. Notebook execution does not change the manuscript or its ZIP automatically.


In [ ]:
from hashlib import sha256

l2 = sweep_plot.loc[sweep_plot["metric"].eq("l2")].copy()
assert len(l2) == 200
assert (l2["n_pairs"] == 10).all()
assert (l2["n_observations"] == 10 * l2["n_instances"]).all()
assert (l2["percent"] == 100 * l2["n_exceed"] / l2["n_observations"]).all()

rows = []
for dataset in ["german", "diabetes", "acs", "gmsc"]:
    for model in ["mlp", "ftt"]:
        group = l2.loc[l2["dataset"].eq(dataset) & l2["model"].eq(model)]
        row = {"dataset": dataset, "model": model,
               "dataset_label": group["dataset_label"].iloc[0],
               "model_label": group["model_label"].iloc[0]}
        for k in [25, 50]:
            subset = group.loc[group["bg_size"].eq(k)]
            assert len(subset) == subset["method"].nunique() == 4
            row[f"K{k}_min_percent"] = float(subset["percent"].min())
            row[f"K{k}_max_percent"] = float(subset["percent"].max())
        large = group.loc[group["bg_size"].ge(100)]
        assert large.groupby("bg_size")["method"].nunique().eq(4).all()
        row["K_ge_100_max_percent"] = float(large["percent"].max())
        row["K_ge_100_configurations"] = len(large)
        row["K_ge_100_nonzero_configurations"] = int(large["n_exceed"].gt(0).sum())
        row["evaluated_K_ge_100"] = ";".join(map(str, sorted(large["bg_size"].unique())))
        rows.append(row)
compact = pd.DataFrame(rows)
assert len(compact) == 8

def range_text(low, high):
    return f"{low:.2f}" if low == high else f"{low:.2f}--{high:.2f}"

latex_lines = [r"\begin{tabular}{llrrr}", r"\toprule",
               r"Dataset & Model & $K=25$ & $K=50$ & Max., $K\geq100$ \\",
               r"\midrule"]
for row in compact.to_dict("records"):
    cells = [row["dataset_label"], row["model_label"],
             range_text(row["K25_min_percent"], row["K25_max_percent"]),
             range_text(row["K50_min_percent"], row["K50_max_percent"]),
             f'{row["K_ge_100_max_percent"]:.2f}']
    latex_lines.append(" & ".join(cells) + r" \\")
latex_lines += [r"\bottomrule", r"\end{tabular}"]

csv_path = OUTPUT / "data/sweep_l2_compact.csv"
tex_path = OUTPUT / "data/sweep_l2_compact.tex"
compact.to_csv(csv_path, index=False, float_format="%.17g")
tex_path.write_text("\n".join(latex_lines) + "\n")
source_path = REPO / "figures_b/background_sweep/pair_exceedance_counts.csv"
summary = {
    "source": str(source_path.relative_to(REPO)),
    "source_sha256": sha256(source_path.read_bytes()).hexdigest(),
    "source_l2_rows": len(l2), "table_rows": len(compact),
    "statistic": "100 * count(distance > saved threshold) / (10N)",
    "K25_K50": "min--max across four sampling methods; not confidence intervals",
    "K_ge_100": "maximum across four methods and every evaluated K >= 100",
    "largest_K_ge_100_percent": float(compact["K_ge_100_max_percent"].max()),
    "all_zero_cells_are_exact_zero_exceedance": True,
    "csv_sha256": sha256(csv_path.read_bytes()).hexdigest(),
    "tex_sha256": sha256(tex_path.read_bytes()).hexdigest(),
    "new_experiments": False,
}
# At this display precision no positive table endpoint is printed as zero.
for column in ["K25_min_percent", "K25_max_percent", "K50_min_percent", "K50_max_percent", "K_ge_100_max_percent"]:
    assert all(value == 0 or round(value, 2) > 0 for value in compact[column])
(OUTPUT / "data/sweep_l2_compact_validation.json").write_text(json.dumps(summary, indent=2) + "\n")
display(compact[["dataset_label", "model_label", "K25_min_percent", "K25_max_percent",
                 "K50_min_percent", "K50_max_percent", "K_ge_100_max_percent"]].round(2))
print("Saved compact table:", tex_path)
print("Largest rate at K>=100:", summary["largest_K_ge_100_percent"], "%")
